# 02 — Feature Creation

Computes every feature used downstream: fixation/saccade metrics per eye/exploration, and GMM/HDA state-based metrics (Fractional Occupancy, Mean Lifetime, Mean Interval Length, entropy) with the BIC curves used to characterise `k`.

Independently runnable: loads `patient_data`/`group_labels` from `../pkls/basic/` (already post-exclusion, produced by `01_exploratory_data_analysis.ipynb`) rather than assuming a shared kernel.

Part of the pipeline: 01 exploratory data analysis → **02 (this notebook)** → 03 feature analysis → 04 model training → 05 model testing.


In [1]:
# Utilities
import os
import random
import pickle

# Data management
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
%matplotlib inline

# Modeling
from sklearn.mixture import GaussianMixture

# Custom utilities
from futils import (
    get_saccade_metrics,
    get_fixation_metrics,
    get_area_metrics,
    calculate_state_standard_metrics,
    entropy,
)


In [2]:
global_random_state = 42

run_metrics_gmm = False  # recompute GMM/HDA state features instead of loading pkls -- takes about 8 hours


In [3]:
with open('../pkls/basic/patient_data.pkl', 'rb') as f:
    patient_data = pickle.load(f)
with open('../pkls/basic/group_labels.pkl', 'rb') as f:
    group_labels = pickle.load(f)

young_control_patients = group_labels[group_labels['group'] == 'young control']['patient_id'].tolist()
control_patients = group_labels[group_labels['group'] == 'control']['patient_id'].tolist()
pd_patients = group_labels[group_labels['group'] == 'PD']['patient_id'].tolist()

# Same seeded held-out split as 01_exploratory_data_analysis.ipynb -- the GMM
# reference model below is fit on control data excluding these patients.
random.seed(42)
control_test = random.sample(control_patients, int(len(control_patients) * 0.2))
pd_test = random.sample(pd_patients, int(len(pd_patients) * 0.2))
test_patients = control_test + pd_test


## Fixation/Saccade-Based Features

For each eye and each exploration: total saccades, total saccadic excursion, total fixations, average fixation time, total scanned area (convex hull), longest diagonal.

$$
d_{i} = || (x_{i}^{0} , y_{i}^{0}) , (x_{i}^{1} , y_{i}^{1}) ||
$$

is the distance for saccadic event $i$, and total saccadic excursion is $\sum_i d_i$.


In [4]:
# Create the metrics dataframes (can be created for all patients in one go since there is no modelling involved)
saccade_metrics_df = get_saccade_metrics(patient_data)
fixation_metrics_df = get_fixation_metrics(patient_data)
area_metrics_df = get_area_metrics(patient_data)

# Join everything in one dataframe
sacfix_metrics = pd.merge(saccade_metrics_df, fixation_metrics_df, on=['patient_id', 'exploration_id', 'eye'], how='outer')
sacfix_metrics = pd.merge(sacfix_metrics, area_metrics_df, on=['patient_id', 'exploration_id', 'eye'], how='outer')
sacfix_metrics = pd.merge(sacfix_metrics, group_labels[['patient_id', 'group', 'group_general']], on='patient_id', how='left')

os.makedirs('../pkls/metrics/', exist_ok=True)
with open('../pkls/metrics/fixation_saccade_metrics.pkl', 'wb') as f:
    pickle.dump(sacfix_metrics, f)

sacfix_metrics.head()


<local path> - Universidad Politécnica de Madrid\Documentos\UPM\repositories\VOG_PD_ML\notebooks\futils.py:321: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  saccades_excursion_df = pd.concat([saccades_excursion_df, new_row], ignore_index=True)


<local path> - Universidad Politécnica de Madrid\Documentos\UPM\repositories\VOG_PD_ML\notebooks\futils.py:357: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fixation_analysis_df = pd.concat([fixation_analysis_df, new_row], ignore_index=True)


<local path> - Universidad Politécnica de Madrid\Documentos\UPM\repositories\VOG_PD_ML\notebooks\futils.py:392: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scanned_area_df = pd.concat([scanned_area_df, new_row], ignore_index=True)


[output removed: contained participant-level rows. Re-run the notebook against your own copy of the recordings to regenerate it.]

## State-Based (HDA) Features

A GMM is fit to the gaze coordinates (`k` = 5 to 50 components) on healthy-control, non-held-out data, defining the High-Density Areas. Each participant's gaze samples are then assigned to components via posterior probability, and Fractional Occupancy (FO), Mean Lifetime (MLT), Mean Interval Length (MIL), and state-transition entropy are computed per patient/eye/exploration/state.


In [5]:
max_states = 50

eyes = ['lts_noblinks', 'rts_noblinks']
explorations = ['Exploration_1', 'Exploration_2', 'Exploration_3', 'Exploration_4', 'Exploration_5', 'Exploration_6']


In [6]:
if not run_metrics_gmm:
    # Read the precomputed GMM metrics from the pickle file
    with open('../pkls/metrics/gmm_metrics.pkl', 'rb') as f:
        gmm_metrics = pickle.load(f)
    with open('../pkls/metrics/bic_scores_dict.pkl', 'rb') as f:
        bic_scores_dict = pickle.load(f)
else:
    # Initialize DataFrames for metrics
    gmm_metrics = pd.DataFrame(columns=['patient_id', 'exploration_id', 'k_clusters', 'eye', 'state', 'FO', 'MLT', 'MIL'])
    entropies_df = pd.DataFrame(columns=['patient_id', 'exploration_id', 'k_clusters', 'eye', 'state', 'entropy'])

    # Prepare for BIC plots
    bic_scores_dict = {exploration: [] for exploration in explorations}

    # Prepare storage for GMM model parameters
    gmm_model_storage = {}

    # Create a 2x3 grid for BIC plots
    fig_bic, axes_bic = plt.subplots(2, 3, figsize=(18, 12))
    exploration_counter = 0

    for exploration in explorations:
        print(f"Processing Exploration: {exploration}")

        for k in range(5, max_states + 1):
            print(f"Processing k={k} for {exploration}")

            all_data = []
            for patient_id in control_patients:
                if patient_id not in test_patients:
                    for eye in eyes:
                        try:
                            patient_eye_data = patient_data[patient_id][exploration][eye].copy()
                            all_data.append(patient_eye_data)
                        except KeyError:
                            continue

            combined_data = pd.concat(all_data)
            combined_data = combined_data[['x', 'y']].dropna().values

            gmm_model = GaussianMixture(
                n_components=k, init_params='kmeans', covariance_type='full', random_state=global_random_state
            )
            gmm_model.fit(combined_data)

            gmm_model_storage[f"{exploration}_k_{k}"] = {
                "means": gmm_model.means_,
                "covariances": gmm_model.covariances_,
                "weights": gmm_model.weights_
            }

            bic_scores_dict[exploration].append(gmm_model.bic(combined_data))

            # GMM component plot
            plt.figure(figsize=(6, 6))
            ax = plt.gca()
            for idx, (mean, covar) in enumerate(zip(gmm_model.means_, gmm_model.covariances_)):
                eigenvalues, eigenvectors = np.linalg.eigh(covar)
                angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
                width, height = 2 * np.sqrt(eigenvalues)
                ellip = Ellipse(xy=mean, width=width, height=height, angle=angle,
                                 edgecolor='black', facecolor='none', lw=1)
                ax.add_patch(ellip)
                ax.text(mean[0], mean[1], f"{idx + 1}", fontsize=14, ha='center', va='center', color='black')
            ax.set_xlim(0, 1)
            ax.set_ylim(0.6, 0)
            ax.set_yticks(np.linspace(0, 1, 6))
            ax.set_yticklabels([f"{tick:.2f}" for tick in np.linspace(1, 0, 6)])
            ax.set_title(f"GMM Components for {exploration.replace('_', ' ')}, k = {k}", fontsize=14)
            ax.set_xlabel("X")
            ax.set_ylabel("Y")
            plt.grid(False)
            plt.tight_layout()
            os.makedirs('../plots/gmm/', exist_ok=True)
            plt.savefig(f"../plots/gmm/GMM_{exploration}_k_{k}_states.png", transparent=True)
            plt.close()

            for patient_id in patient_data.keys():
                for eye in ['lts_noblinks', 'rts_noblinks']:
                    try:
                        patient_eye_data = patient_data[patient_id][exploration][eye].copy()
                    except KeyError:
                        continue

                    patient_eye_data = patient_eye_data[['x', 'y']].dropna()
                    patient_eye_data['cluster'] = gmm_model.predict(patient_eye_data[['x', 'y']].values)
                    viterbi_path = patient_eye_data['cluster'].values
                    metrics = calculate_state_standard_metrics(viterbi_path)
                    metrics['patient_id'] = patient_id
                    metrics['exploration_id'] = exploration
                    metrics['k_clusters'] = k
                    metrics['eye'] = eye
                    gmm_metrics = pd.concat([gmm_metrics, metrics], ignore_index=True)

                    transitions_matrix = np.zeros((k, k))
                    for i in range(len(viterbi_path) - 1):
                        transitions_matrix[viterbi_path[i], viterbi_path[i + 1]] += 1

                    row_sums = transitions_matrix.sum(axis=1, keepdims=True)
                    row_sums[row_sums == 0] = 1
                    transitions_matrix = transitions_matrix / row_sums

                    state_entropies = np.apply_along_axis(entropy, 1, transitions_matrix)
                    for state_idx, entropy_value in enumerate(state_entropies):
                        entropies_df = pd.concat([entropies_df, pd.DataFrame([{
                            'patient_id': patient_id,
                            'exploration_id': exploration,
                            'k_clusters': k,
                            'eye': eye,
                            'state': state_idx,
                            'entropy': entropy_value,
                        }])], ignore_index=True)

        ax_bic = axes_bic[exploration_counter // 3, exploration_counter % 3]
        ax_bic.plot(range(5, max_states + 1), bic_scores_dict[exploration], marker='o', color='#556B2F')
        ax_bic.set_title(f"{exploration.replace('_', ' ')}", fontsize=14)
        ax_bic.set_xlabel("k")
        ax_bic.set_ylabel("BIC Score")
        ax_bic.grid(True)
        exploration_counter += 1

    fig_bic.tight_layout()
    fig_bic.suptitle("Gaussian Mixture Models - BIC", fontsize=16, fontweight='bold', y=1.02)
    plt.savefig("../plots/gmm/BIC_Scores_All_Explorations.png")
    plt.show()

    # Merge metrics and save
    gmm_metrics = pd.merge(
        gmm_metrics, entropies_df,
        on=['patient_id', 'exploration_id', 'k_clusters', 'eye', 'state'], how='outer',
    ).fillna(0)
    gmm_metrics = pd.merge(gmm_metrics, group_labels, on='patient_id')
    gmm_metrics['eye'] = gmm_metrics['eye'].replace({'lts_noblinks': 'left', 'rts_noblinks': 'right'})

    with open('../pkls/metrics/gmm_metrics.pkl', 'wb') as f:
        pickle.dump(gmm_metrics, f)
    with open('../pkls/metrics/bic_scores_dict.pkl', 'wb') as f:
        pickle.dump(bic_scores_dict, f)
    with open("../pkls/metrics/gmm_model_storage.pkl", "wb") as f:
        pickle.dump(gmm_model_storage, f)

gmm_metrics.head()


[output removed: contained participant-level rows. Re-run the notebook against your own copy of the recordings to regenerate it.]